# Wilder RSI tenor-isolated parameter sweep v1

**Purpose:** optimize the repaired-Wilder-RSI Core and Secondary thresholds for each tenor independently.

This notebook is self-contained. It does not call an external `.py` script.

## Sources

- Repaired signal base:
  `data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet`
- Historical put outcomes:
  latest `07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet` under
  `data\processed\vrp_front_middle_corsi_forecast_repair_v1`

The historical outcome panel is the same source family used by the prior cleaned threshold-research notebook. This notebook imports only realized outcome fields from it:

- `normalized_pnl_pct`
- `normalized_pnl_dollars`
- `win`
- `last_forward_rv_date`

It does **not** use the legacy `RSI14` field from that outcome source. Signal gating uses only the repaired `rsi14_final` and its accepted formula version from the repaired signal base.

## Fixed scope

- Tenors: 12D, 15D, 18D, 21D, 24D, 27D, 30D, 33D
- 9D inactive
- Core Front inactive
- Core and Secondary tested separately
- Each tenor evaluated independently
- Every qualifying tenor counts
- No one-trade-per-day suppression
- No sizing
- No production overwrite
- Strict comparisons: `>` for VRP/z/RV21D and `<` for RSI
- 3m and 1y z thresholds tied within each tested parameter set

## Parameter grid

For each active tenor/layer, centered on the approved starting parameters:

- VRP log: baseline ±0.10, step 0.05
- tied z threshold: baseline ±0.20, step 0.10
- RSI cap: baseline ±5, step 1
- RV21D floor: baseline ±1.5, step 0.5

This produces 1,925 parameter sets per sleeve and 25,025 total results.

## Ranking

Parameter sets with at least 30 trades are ranked within each tenor/layer using an equal-weight percentile composite of:

- win rate
- average normalized return
- profit factor
- worst 1% expected return
- additive return drawdown
- largest consecutive losing cluster

Trade count and frequency are reported but are not rewarded directly in the quality score.

In [1]:
# ======================================================================================
# Cell 0 — Setup and fixed research contract
# ======================================================================================
#
# This notebook replaces the external-script/subprocess workflow.
# It follows the structure and outcome conventions of the prior cleaned threshold notebook.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import itertools
import json
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 520)
pd.set_option("display.max_rows", 200)

# Allow a test harness or user to override PROJECT_ROOT before running this cell.
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", r"C:\Users\patri\vrp_project"))

SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_repaired_wilder_rsi_signal"
    / "vrp_repaired_wilder_rsi_signal_base_v1.parquet"
)

OUTCOME_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_front_middle_corsi_forecast_repair_v1"
)
OUTCOME_PATTERN = "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet"

OUT_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_wilder_rsi_parameter_sweep"
)
OUT_AUDIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "audit"
    / "wilder_rsi_tenor_isolated_parameter_sweep"
)

OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = (
    OUT_PROCESSED_DIR
    / "vrp_wilder_rsi_tenor_isolated_parameter_sweep_results_v1.parquet"
)
LEADERBOARD_PATH = (
    OUT_PROCESSED_DIR
    / "vrp_wilder_rsi_tenor_isolated_parameter_leaderboard_v1.parquet"
)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"
ACCEPTED_RSI_VERSION = "wilder_rsi14_spy_close_v2_long_warmup"

TARGET_TENORS = [12, 15, 18, 21, 24, 27, 30, 33]
MIN_RANKING_TRADES = 30
TOP_PER_SLEEVE = 25
TAIL_FRACTION = 0.05
WORST_FRACTION = 0.01

# Core Front and 9D are intentionally absent.
BASELINES = {
    ("Core", 21): {"vrp_log": 0.65, "z": 0.70, "rsi_cap": 70.0, "rv_floor": 8.5},
    ("Core", 24): {"vrp_log": 0.65, "z": 0.70, "rsi_cap": 70.0, "rv_floor": 8.5},
    ("Core", 27): {"vrp_log": 0.70, "z": 0.70, "rsi_cap": 70.0, "rv_floor": 8.5},
    ("Core", 30): {"vrp_log": 0.70, "z": 0.70, "rsi_cap": 70.0, "rv_floor": 8.5},
    ("Core", 33): {"vrp_log": 0.70, "z": 0.70, "rsi_cap": 70.0, "rv_floor": 8.5},
    ("Secondary", 12): {"vrp_log": 0.65, "z": 0.20, "rsi_cap": 75.0, "rv_floor": 7.0},
    ("Secondary", 15): {"vrp_log": 0.65, "z": 0.20, "rsi_cap": 75.0, "rv_floor": 7.0},
    ("Secondary", 18): {"vrp_log": 0.65, "z": 0.20, "rsi_cap": 75.0, "rv_floor": 7.0},
    ("Secondary", 21): {"vrp_log": 0.65, "z": 0.20, "rsi_cap": 76.0, "rv_floor": 7.0},
    ("Secondary", 24): {"vrp_log": 0.65, "z": 0.20, "rsi_cap": 76.0, "rv_floor": 7.0},
    ("Secondary", 27): {"vrp_log": 0.65, "z": 0.00, "rsi_cap": 77.0, "rv_floor": 6.5},
    ("Secondary", 30): {"vrp_log": 0.65, "z": 0.00, "rsi_cap": 77.0, "rv_floor": 6.5},
    ("Secondary", 33): {"vrp_log": 0.65, "z": 0.00, "rsi_cap": 77.0, "rv_floor": 6.5},
}

VRP_OFFSETS = [-0.10, -0.05, 0.00, 0.05, 0.10]
Z_OFFSETS = [-0.20, -0.10, 0.00, 0.10, 0.20]
RSI_OFFSETS = list(range(-5, 6))
RV_OFFSETS = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

EXPECTED_GRID_SIZE_PER_SLEEVE = (
    len(VRP_OFFSETS) * len(Z_OFFSETS) * len(RSI_OFFSETS) * len(RV_OFFSETS)
)
EXPECTED_RESULT_ROWS = len(BASELINES) * EXPECTED_GRID_SIZE_PER_SLEEVE

if len(BASELINES) != 13:
    raise RuntimeError(f"Expected 13 active tenor/layer sleeves, found {len(BASELINES)}")
if EXPECTED_GRID_SIZE_PER_SLEEVE != 1925:
    raise RuntimeError(
        f"Expected 1,925 parameter sets per sleeve, found {EXPECTED_GRID_SIZE_PER_SLEEVE}"
    )

print("=" * 100)
print("Cell 0 — Setup and fixed research contract")
print("=" * 100)
print(f"Project root:                 {PROJECT_ROOT}")
print(f"Repaired signal source:       {SIGNAL_PATH}")
print(f"Historical outcome directory: {OUTCOME_DIR}")
print(f"Historical outcome pattern:   {OUTCOME_PATTERN}")
print(f"Processed output directory:   {OUT_PROCESSED_DIR}")
print(f"Audit output directory:       {OUT_AUDIT_DIR}")
print(f"Target tenors:                {TARGET_TENORS}")
print(f"Active sleeves:               {len(BASELINES)}")
print(f"Grid per sleeve:              {EXPECTED_GRID_SIZE_PER_SLEEVE:,}")
print(f"Expected result rows:          {EXPECTED_RESULT_ROWS:,}")
print()
print("Selection rule: none — all qualifying tenor opportunities are preserved.")
print("Core Front and 9D remain inactive.")
print("No sizing and no production writes occur in this notebook.")
print("\nCELL 0 COMPLETE")

Cell 0 — Setup and fixed research contract
Project root:                 C:\Users\patri\vrp_project
Repaired signal source:       C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet
Historical outcome directory: C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1
Historical outcome pattern:   07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet
Processed output directory:   C:\Users\patri\vrp_project\data\processed\vrp_wilder_rsi_parameter_sweep
Audit output directory:       C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep
Target tenors:                [12, 15, 18, 21, 24, 27, 30, 33]
Active sleeves:               13
Grid per sleeve:              1,925
Expected result rows:          25,025

Selection rule: none — all qualifying tenor opportunities are preserved.
Core Front and 9D remain inactive.
No sizing and no production writes occur in this notebo

In [2]:
# ======================================================================================
# Cell 1 — Load repaired signal base and established historical outcome panel
# ======================================================================================
#
# Historical outcome source:
#   latest 07A unified_fds_no_min_return OOS forecast panel
#
# Imported outcome fields only:
#   normalized_pnl_pct
#   normalized_pnl_dollars
#   win
#   last_forward_rv_date
#
# Legacy RSI14 from the historical outcome panel is not loaded or used.
# ======================================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(
        directory.glob(pattern),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not matches:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return matches[0]


def parse_date(
    series: pd.Series,
    label: str,
    allow_null: bool = False,
) -> pd.Series:
    raw = series.copy()
    numeric_text = raw.astype("string").str.replace(r"\.0$", "", regex=True)
    yyyymmdd_mask = numeric_text.str.fullmatch(r"\d{8}", na=False)
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    if yyyymmdd_mask.any():
        parsed.loc[yyyymmdd_mask] = pd.to_datetime(
            numeric_text.loc[yyyymmdd_mask],
            format="%Y%m%d",
            errors="coerce",
        )
    if (~yyyymmdd_mask).any():
        parsed.loc[~yyyymmdd_mask] = pd.to_datetime(
            raw.loc[~yyyymmdd_mask],
            errors="coerce",
        )
    parsed = parsed.dt.normalize()
    bad_mask = parsed.isna() & raw.notna()
    if bad_mask.any():
        bad = raw.loc[bad_mask].head(10).tolist()
        raise ValueError(f"{label} contains unparseable dates. Examples: {bad}")
    if not allow_null and parsed.isna().any():
        raise ValueError(f"{label} contains null dates.")
    return parsed


def parse_tenor(series: pd.Series, label: str) -> pd.Series:
    tenor = pd.to_numeric(series, errors="coerce")
    if tenor.isna().any():
        bad = series.loc[tenor.isna()].head(10).tolist()
        raise ValueError(f"{label} contains non-numeric values. Examples: {bad}")
    non_integer = (tenor - tenor.round()).abs() > 1e-12
    if non_integer.any():
        bad = tenor.loc[non_integer].head(10).tolist()
        raise ValueError(f"{label} contains non-integer values. Examples: {bad}")
    return tenor.round().astype(int)


def require_columns(df: pd.DataFrame, required: list[str], label: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{label} missing required columns: {missing}")


def additive_drawdown(values: pd.Series) -> float:
    x = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    cumulative = np.concatenate(([0.0], np.cumsum(x)))
    running_peak = np.maximum.accumulate(cumulative)
    return float(np.min(cumulative - running_peak))


if not SIGNAL_PATH.exists():
    raise FileNotFoundError(f"Repaired signal base does not exist: {SIGNAL_PATH}")
if not OUTCOME_DIR.exists():
    raise FileNotFoundError(f"Historical outcome directory does not exist: {OUTCOME_DIR}")

OUTCOME_PATH = latest_file(OUTCOME_DIR, OUTCOME_PATTERN)

signal_raw = pd.read_parquet(SIGNAL_PATH)
outcome_raw = pd.read_parquet(OUTCOME_PATH)

required_signal_cols = [
    "trade_date",
    "tenor",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
    "rv21d_vol_pct_final",
    "rsi_formula_version",
]
required_outcome_cols = [
    "trade_date",
    "tenor",
    "model_spec",
    "normalized_pnl_pct",
    "normalized_pnl_dollars",
    "win",
    "last_forward_rv_date",
]

require_columns(signal_raw, required_signal_cols, "Repaired signal base")
require_columns(outcome_raw, required_outcome_cols, "Historical outcome panel")

signal = pd.DataFrame({
    "trade_date": parse_date(signal_raw["trade_date"], "signal trade_date"),
    "tenor": parse_tenor(signal_raw["tenor"], "signal tenor"),
    "model_vrp_log": pd.to_numeric(signal_raw["model_vrp_log_final"], errors="coerce"),
    "z_3m": pd.to_numeric(signal_raw["z_3m_final"], errors="coerce"),
    "z_1y": pd.to_numeric(signal_raw["z_1y_final"], errors="coerce"),
    "rsi14": pd.to_numeric(signal_raw["rsi14_final"], errors="coerce"),
    "rv21d_vol_pct": pd.to_numeric(signal_raw["rv21d_vol_pct_final"], errors="coerce"),
    "rsi_formula_version": signal_raw["rsi_formula_version"].astype("string"),
})

signal = signal.loc[signal["tenor"].isin(TARGET_TENORS)].copy()

if signal.duplicated(["trade_date", "tenor"]).any():
    dup = signal.loc[
        signal.duplicated(["trade_date", "tenor"], keep=False),
        ["trade_date", "tenor"],
    ].head(20)
    display(dup)
    raise RuntimeError("Repaired signal base is not unique on trade_date × tenor.")

signal_versions = sorted(signal["rsi_formula_version"].dropna().unique().tolist())
if signal_versions != [ACCEPTED_RSI_VERSION]:
    raise RuntimeError(
        "Unexpected repaired RSI formula version(s): "
        f"{signal_versions}; expected only {ACCEPTED_RSI_VERSION!r}"
    )

outcome_filtered = outcome_raw.loc[
    outcome_raw["model_spec"].astype(str).eq(LOCKED_MODEL_SPEC)
].copy()

outcome = pd.DataFrame({
    "trade_date": parse_date(outcome_filtered["trade_date"], "outcome trade_date"),
    "tenor": parse_tenor(outcome_filtered["tenor"], "outcome tenor"),
    "last_forward_rv_date": parse_date(
        outcome_filtered["last_forward_rv_date"],
        "outcome last_forward_rv_date",
        allow_null=True,
    ),
    "normalized_pnl_pct": pd.to_numeric(
        outcome_filtered["normalized_pnl_pct"],
        errors="coerce",
    ),
    "normalized_pnl_dollars": pd.to_numeric(
        outcome_filtered["normalized_pnl_dollars"],
        errors="coerce",
    ),
    "win": pd.to_numeric(outcome_filtered["win"], errors="coerce"),
})

outcome = outcome.loc[outcome["tenor"].isin(TARGET_TENORS)].copy()

if outcome.duplicated(["trade_date", "tenor"]).any():
    dup = outcome.loc[
        outcome.duplicated(["trade_date", "tenor"], keep=False),
        ["trade_date", "tenor", "normalized_pnl_pct", "normalized_pnl_dollars", "win"],
    ].sort_values(["trade_date", "tenor"]).head(30)
    display(dup)
    raise RuntimeError(
        "Filtered unified-FDS historical outcome panel is not unique on trade_date × tenor."
    )

missing_signal_tenors = sorted(set(TARGET_TENORS) - set(signal["tenor"].unique()))
missing_outcome_tenors = sorted(set(TARGET_TENORS) - set(outcome["tenor"].unique()))
if missing_signal_tenors:
    raise RuntimeError(f"Repaired signal base missing target tenors: {missing_signal_tenors}")
if missing_outcome_tenors:
    raise RuntimeError(f"Historical outcome panel missing target tenors: {missing_outcome_tenors}")

max_outcome_panel_date = outcome["trade_date"].max()
outcome["forward_window_complete"] = (
    outcome["last_forward_rv_date"].notna()
    & (outcome["last_forward_rv_date"] <= max_outcome_panel_date)
)
outcome["outcome_complete"] = (
    outcome["forward_window_complete"]
    & outcome["normalized_pnl_pct"].notna()
    & outcome["normalized_pnl_dollars"].notna()
    & outcome["win"].notna()
)

# Confirm the established win field matches the normalized-return sign.
complete_for_win_check = outcome.loc[outcome["outcome_complete"]].copy()
complete_for_win_check["expected_win_from_return"] = (
    complete_for_win_check["normalized_pnl_pct"] > 0
).astype(float)
win_mismatch = complete_for_win_check.loc[
    complete_for_win_check["win"] != complete_for_win_check["expected_win_from_return"]
]
if len(win_mismatch):
    display(win_mismatch.head(20))
    raise RuntimeError(
        f"Historical outcome win field disagrees with normalized_pnl_pct sign on "
        f"{len(win_mismatch):,} rows."
    )

outcome_complete = outcome.loc[outcome["outcome_complete"]].copy()

joined = signal.merge(
    outcome_complete[
        [
            "trade_date",
            "tenor",
            "last_forward_rv_date",
            "normalized_pnl_pct",
            "normalized_pnl_dollars",
            "win",
        ]
    ],
    on=["trade_date", "tenor"],
    how="inner",
    validate="one_to_one",
)

signal_input_cols = [
    "model_vrp_log",
    "z_3m",
    "z_1y",
    "rsi14",
    "rv21d_vol_pct",
]
joined["signal_inputs_complete"] = joined[signal_input_cols].notna().all(axis=1)
joined_usable = (
    joined.loc[joined["signal_inputs_complete"]]
    .sort_values(["tenor", "trade_date"])
    .reset_index(drop=True)
)

if joined_usable.empty:
    raise RuntimeError("No rows have both repaired signal inputs and complete outcomes.")

missing_joined_tenors = sorted(
    set(TARGET_TENORS) - set(joined_usable["tenor"].unique())
)
if missing_joined_tenors:
    raise RuntimeError(
        f"No usable joined rows for target tenors: {missing_joined_tenors}"
    )

# Require complete outcome coverage inside each tenor's usable realized window.
coverage_failures = []
coverage_rows = []

for tenor in TARGET_TENORS:
    sig_t = signal.loc[signal["tenor"] == tenor].copy()
    out_t = outcome_complete.loc[outcome_complete["tenor"] == tenor].copy()
    usable_t = joined_usable.loc[joined_usable["tenor"] == tenor].copy()

    first_outcome_date = out_t["trade_date"].min()
    last_outcome_date = out_t["trade_date"].max()

    expected_signal_t = sig_t.loc[
        (sig_t["trade_date"] >= first_outcome_date)
        & (sig_t["trade_date"] <= last_outcome_date)
        & sig_t[signal_input_cols].notna().all(axis=1)
    ]

    missing_keys = expected_signal_t[["trade_date", "tenor"]].merge(
        out_t[["trade_date", "tenor"]],
        on=["trade_date", "tenor"],
        how="left",
        indicator=True,
    )
    missing_keys = missing_keys.loc[missing_keys["_merge"] == "left_only"]

    if len(missing_keys):
        for row in missing_keys.head(10).itertuples(index=False):
            coverage_failures.append({
                "trade_date": row.trade_date,
                "tenor": int(row.tenor),
            })

    coverage_rows.append({
        "tenor": tenor,
        "signal_rows": len(sig_t),
        "signal_first_date": sig_t["trade_date"].min(),
        "signal_last_date": sig_t["trade_date"].max(),
        "complete_outcome_rows": len(out_t),
        "outcome_first_date": first_outcome_date,
        "outcome_last_date": last_outcome_date,
        "usable_joined_rows": len(usable_t),
        "expected_complete_signal_rows_in_outcome_window": len(expected_signal_t),
        "missing_outcome_keys_in_window": len(missing_keys),
    })

if coverage_failures:
    raise RuntimeError(
        "Historical outcome coverage has gaps inside a usable realized window. "
        f"Examples: {coverage_failures[:20]}"
    )

join_coverage = pd.DataFrame(coverage_rows)

max_tenors_per_outcome_date = int(
    outcome_complete.groupby("trade_date")["tenor"].nunique().max()
)
if max_tenors_per_outcome_date <= 1:
    raise RuntimeError(
        "Historical outcome source appears one-trade-per-day filtered."
    )

print("=" * 100)
print("Cell 1 — Loaded repaired signals and established historical outcomes")
print("=" * 100)
print(f"Repaired signal source:       {SIGNAL_PATH}")
print(f"Historical outcome source:    {OUTCOME_PATH}")
print(f"Signal raw rows/columns:      {len(signal_raw):,} / {len(signal_raw.columns):,}")
print(f"Outcome raw rows/columns:     {len(outcome_raw):,} / {len(outcome_raw.columns):,}")
print(f"Unified-FDS outcome rows:     {len(outcome):,}")
print(f"Complete realized outcomes:   {len(outcome_complete):,}")
print(f"Joined usable rows:           {len(joined_usable):,}")
print(f"Signal date range:            {signal['trade_date'].min().date()} to {signal['trade_date'].max().date()}")
print(f"Outcome usable date range:    {outcome_complete['trade_date'].min().date()} to {outcome_complete['trade_date'].max().date()}")
print(f"Max tenors per outcome date:  {max_tenors_per_outcome_date}")
print(f"RSI formula version:          {signal_versions[0]}")
print()
print("Historical fields imported:")
print("  normalized_pnl_pct")
print("  normalized_pnl_dollars")
print("  win")
print("  last_forward_rv_date")
print()
print("Legacy RSI14 from the historical outcome panel was not loaded.")
print("\nJoin coverage by tenor:")
display(join_coverage)

print("\nCELL 1 COMPLETE")

Cell 1 — Loaded repaired signals and established historical outcomes
Repaired signal source:       C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet
Historical outcome source:    C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260709_20260710_101156_20260710_100854_schema_repair.parquet
Signal raw rows/columns:      14,724 / 27
Outcome raw rows/columns:     31,185 / 90
Unified-FDS outcome rows:     13,096
Complete realized outcomes:   12,879
Joined usable rows:           10,863
Signal date range:            2020-01-02 to 2026-07-09
Outcome usable date range:    2020-01-02 to 2026-06-09
Max tenors per outcome date:  8
RSI formula version:          wilder_rsi14_spy_close_v2_long_warmup

Historical fields imported:
  normalized_pnl_pct
  normalized_pnl_dollars
  win
  last_forward_rv_date

Legacy RSI14 from the historical out

,tenor,signal_rows,signal_first_date,signal_last_date,complete_outcome_rows,outcome_first_date,outcome_last_date,usable_joined_rows,expected_complete_signal_rows_in_outcome_window,missing_outcome_keys_in_window
0,12,1636,2020-01-02,2026-07-09,1617,2020-01-02,2026-06-09,1365,1365,0
1,15,1636,2020-01-02,2026-07-09,1615,2020-01-02,2026-06-05,1363,1363,0
2,18,1636,2020-01-02,2026-07-09,1613,2020-01-02,2026-06-03,1361,1361,0
3,21,1636,2020-01-02,2026-07-09,1610,2020-01-02,2026-05-29,1358,1358,0
4,24,1636,2020-01-02,2026-07-09,1609,2020-01-02,2026-05-28,1357,1357,0
5,27,1636,2020-01-02,2026-07-09,1606,2020-01-02,2026-05-22,1354,1354,0
6,30,1636,2020-01-02,2026-07-09,1606,2020-01-02,2026-05-22,1354,1354,0
7,33,1636,2020-01-02,2026-07-09,1603,2020-01-02,2026-05-19,1351,1351,0



CELL 1 COMPLETE


In [3]:
# ======================================================================================
# Cell 2 — Parameter-grid, performance, tail, drawdown, and ranking helpers
# ======================================================================================

def rounded_grid(baseline: float, offsets: list[float], decimals: int = 10) -> list[float]:
    return sorted({round(float(baseline) + float(x), decimals) for x in offsets})


def build_parameter_grid(baseline: dict[str, float]) -> list[dict[str, float]]:
    vrp_values = rounded_grid(baseline["vrp_log"], VRP_OFFSETS)
    z_values = rounded_grid(baseline["z"], Z_OFFSETS)
    rsi_values = rounded_grid(baseline["rsi_cap"], RSI_OFFSETS)
    rv_values = rounded_grid(baseline["rv_floor"], RV_OFFSETS)

    grid = [
        {
            "vrp_log": vrp,
            "z": z,
            "rsi_cap": rsi,
            "rv_floor": rv,
        }
        for vrp, z, rsi, rv in itertools.product(
            vrp_values,
            z_values,
            rsi_values,
            rv_values,
        )
    ]

    if len(grid) != EXPECTED_GRID_SIZE_PER_SLEEVE:
        raise RuntimeError(
            f"Unexpected parameter-grid size: {len(grid)} "
            f"!= {EXPECTED_GRID_SIZE_PER_SLEEVE}"
        )

    return grid


def worst_expected_shortfall(values: np.ndarray, fraction: float) -> float:
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return np.nan
    count = max(1, int(math.ceil(len(x) * fraction)))
    worst = np.partition(x, count - 1)[:count]
    return float(np.mean(worst))


def max_drawdown_additive(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return np.nan
    wealth = np.concatenate(([0.0], np.cumsum(x)))
    running_peak = np.maximum.accumulate(wealth)
    return float(np.min(wealth - running_peak))


def largest_consecutive_losing_cluster(values: np.ndarray) -> tuple[float, int]:
    worst_sum = 0.0
    worst_count = 0
    current_sum = 0.0
    current_count = 0

    for value in np.asarray(values, dtype=float):
        if value < 0:
            current_sum += float(value)
            current_count += 1
            if current_sum < worst_sum:
                worst_sum = current_sum
                worst_count = current_count
        else:
            current_sum = 0.0
            current_count = 0

    return float(worst_sum), int(worst_count)


def worst_rolling_sum(values: np.ndarray, window: int) -> float:
    x = pd.Series(np.asarray(values, dtype=float))
    if len(x) == 0:
        return np.nan
    if len(x) < window:
        return float(x.sum())
    return float(x.rolling(window=window, min_periods=window).sum().min())


def profit_factor(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    wins = x[x > 0]
    losses = x[x < 0]

    gross_profit = float(wins.sum()) if len(wins) else 0.0
    gross_loss = float(-losses.sum()) if len(losses) else 0.0

    if gross_loss > 0:
        return gross_profit / gross_loss
    if gross_profit > 0:
        return np.inf
    return np.nan


def calculate_metrics(
    selected_returns: np.ndarray,
    selected_pnl_dollars: np.ndarray,
    selected_wins: np.ndarray,
    tenor: int,
    eligible_rows: int,
) -> dict:
    returns = np.asarray(selected_returns, dtype=float)
    pnl = np.asarray(selected_pnl_dollars, dtype=float)
    wins = np.asarray(selected_wins, dtype=float)

    trade_count = int(len(returns))

    if trade_count == 0:
        return {
            "trade_count": 0,
            "frequency": 0.0 if eligible_rows else np.nan,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "median_return": np.nan,
            "avg_win": np.nan,
            "avg_loss": np.nan,
            "max_loss": np.nan,
            "tail_loss": np.nan,
            "worst_1pct_loss": np.nan,
            "profit_factor": np.nan,
            "max_drawdown": np.nan,
            "largest_losing_cluster": np.nan,
            "largest_losing_cluster_count": 0,
            "total_pnl_dollars": 0.0,
            "avg_pnl_dollars": np.nan,
            "avg_pnl_per_day": np.nan,
            "pnl_drawdown_dollars": np.nan,
            "worst_20_trade_pnl_dollars": np.nan,
        }

    positive_returns = returns[returns > 0]
    negative_returns = returns[returns < 0]
    pnl_per_day = pnl / float(tenor)

    cluster_sum, cluster_count = largest_consecutive_losing_cluster(returns)

    return {
        "trade_count": trade_count,
        "frequency": float(trade_count / eligible_rows) if eligible_rows else np.nan,
        "win_rate": float(wins.mean()),
        "avg_return": float(returns.mean()),
        "median_return": float(np.median(returns)),
        "avg_win": float(positive_returns.mean()) if len(positive_returns) else np.nan,
        "avg_loss": float(negative_returns.mean()) if len(negative_returns) else np.nan,
        "max_loss": float(returns.min()),
        "tail_loss": worst_expected_shortfall(returns, TAIL_FRACTION),
        "worst_1pct_loss": worst_expected_shortfall(returns, WORST_FRACTION),
        "profit_factor": profit_factor(returns),
        "max_drawdown": max_drawdown_additive(returns),
        "largest_losing_cluster": cluster_sum,
        "largest_losing_cluster_count": cluster_count,
        "total_pnl_dollars": float(pnl.sum()),
        "avg_pnl_dollars": float(pnl.mean()),
        "avg_pnl_per_day": float(pnl_per_day.mean()),
        "pnl_drawdown_dollars": max_drawdown_additive(pnl),
        "worst_20_trade_pnl_dollars": worst_rolling_sum(pnl, 20),
    }


def percentile_rank_higher_is_better(
    series: pd.Series,
    eligible: pd.Series,
) -> pd.Series:
    result = pd.Series(np.nan, index=series.index, dtype=float)
    values = pd.to_numeric(series.loc[eligible], errors="coerce")

    if values.empty:
        return result

    finite = values.replace([np.inf, -np.inf], np.nan).dropna()

    if finite.empty:
        adjusted = values.replace(np.inf, 1.0).replace(-np.inf, -1.0)
    else:
        finite_max = float(finite.max())
        finite_min = float(finite.min())
        adjusted = values.replace(
            np.inf,
            finite_max + max(1.0, abs(finite_max) * 0.01),
        ).replace(
            -np.inf,
            finite_min - max(1.0, abs(finite_min) * 0.01),
        )

    result.loc[eligible] = adjusted.rank(
        method="average",
        pct=True,
        ascending=True,
    )
    return result


def assign_parameter_ranks(results: pd.DataFrame) -> pd.DataFrame:
    ranking_metrics = [
        "win_rate",
        "avg_return",
        "profit_factor",
        "worst_1pct_loss",
        "max_drawdown",
        "largest_losing_cluster",
    ]

    parts = []

    for (layer, tenor), group in results.groupby(["layer", "tenor"], sort=False):
        g = group.copy()
        eligible = g["trade_count"] >= MIN_RANKING_TRADES

        percentile_cols = []

        for metric in ranking_metrics:
            pct_col = f"rank_pct_{metric}"
            g[pct_col] = percentile_rank_higher_is_better(g[metric], eligible)
            percentile_cols.append(pct_col)

        g["ranking_eligible"] = eligible
        g["parameter_quality_score"] = g[percentile_cols].mean(
            axis=1,
            skipna=False,
        )
        g["parameter_set_rank"] = pd.Series(pd.NA, index=g.index, dtype="Int64")

        if eligible.any():
            ordered = g.loc[eligible].sort_values(
                [
                    "parameter_quality_score",
                    "win_rate",
                    "avg_return",
                    "worst_1pct_loss",
                    "trade_count",
                    "parameter_set_id",
                ],
                ascending=[False, False, False, False, False, True],
                kind="mergesort",
            )
            g.loc[ordered.index, "parameter_set_rank"] = np.arange(
                1,
                len(ordered) + 1,
            )

        parts.append(g)

    return pd.concat(parts, ignore_index=True)


print("=" * 100)
print("Cell 2 — Helpers loaded")
print("=" * 100)
print(f"Grid size per sleeve: {EXPECTED_GRID_SIZE_PER_SLEEVE:,}")
print(f"Tail loss definition: average of worst {TAIL_FRACTION:.0%} returns")
print(f"Worst 1% definition: average of worst {WORST_FRACTION:.0%} returns")
print(f"Ranking minimum:      {MIN_RANKING_TRADES} trades")
print("\nCELL 2 COMPLETE")

Cell 2 — Helpers loaded
Grid size per sleeve: 1,925
Tail loss definition: average of worst 5% returns
Worst 1% definition: average of worst 1% returns
Ranking minimum:      30 trades

CELL 2 COMPLETE


In [4]:
# ======================================================================================
# Cell 3 — Run independent tenor/layer parameter sweep
# ======================================================================================
#
# Every sleeve is evaluated against all complete outcomes for its own tenor.
# No date-level suppression is applied.
# ======================================================================================

result_rows = []
baseline_opportunity_parts = []

for sleeve_number, ((layer, tenor), baseline) in enumerate(BASELINES.items(), start=1):
    tenor_frame = (
        joined_usable.loc[joined_usable["tenor"] == tenor]
        .sort_values("trade_date")
        .reset_index(drop=True)
    )

    eligible_rows = len(tenor_frame)

    vrp = tenor_frame["model_vrp_log"].to_numpy(dtype=float)
    z3 = tenor_frame["z_3m"].to_numpy(dtype=float)
    z1 = tenor_frame["z_1y"].to_numpy(dtype=float)
    rsi = tenor_frame["rsi14"].to_numpy(dtype=float)
    rv = tenor_frame["rv21d_vol_pct"].to_numpy(dtype=float)

    returns = tenor_frame["normalized_pnl_pct"].to_numpy(dtype=float)
    pnl_dollars = tenor_frame["normalized_pnl_dollars"].to_numpy(dtype=float)
    wins = tenor_frame["win"].to_numpy(dtype=float)
    dates = tenor_frame["trade_date"].to_numpy()

    grid = build_parameter_grid(baseline)
    baseline_hits = 0

    for parameter_number, params in enumerate(grid, start=1):
        is_baseline = (
            math.isclose(params["vrp_log"], baseline["vrp_log"], abs_tol=1e-12)
            and math.isclose(params["z"], baseline["z"], abs_tol=1e-12)
            and math.isclose(params["rsi_cap"], baseline["rsi_cap"], abs_tol=1e-12)
            and math.isclose(params["rv_floor"], baseline["rv_floor"], abs_tol=1e-12)
        )
        baseline_hits += int(is_baseline)

        mask = (
            (vrp > params["vrp_log"])
            & (z3 > params["z"])
            & (z1 > params["z"])
            & (rsi < params["rsi_cap"])
            & (rv > params["rv_floor"])
        )

        selected_returns = returns[mask]
        selected_pnl = pnl_dollars[mask]
        selected_wins = wins[mask]
        selected_dates = dates[mask]

        metrics = calculate_metrics(
            selected_returns=selected_returns,
            selected_pnl_dollars=selected_pnl,
            selected_wins=selected_wins,
            tenor=tenor,
            eligible_rows=eligible_rows,
        )

        parameter_set_id = (
            f"{layer.lower()}_{tenor:02d}d_"
            f"vrp{params['vrp_log']:+.2f}_"
            f"z{params['z']:+.2f}_"
            f"rsi{params['rsi_cap']:.0f}_"
            f"rv{params['rv_floor']:.1f}"
        ).replace("+", "p").replace("-", "m").replace(".", "p")

        result_rows.append({
            "parameter_set_id": parameter_set_id,
            "layer": layer,
            "layer_sort": 1 if layer == "Core" else 2,
            "tenor": tenor,
            "vrp_log_threshold": params["vrp_log"],
            "z_3m_threshold": params["z"],
            "z_1y_threshold": params["z"],
            "rsi_cap": params["rsi_cap"],
            "rv21d_floor": params["rv_floor"],
            "is_baseline": is_baseline,
            "eligible_outcome_rows": eligible_rows,
            "selected_first_date": (
                pd.Timestamp(selected_dates.min()).normalize()
                if len(selected_dates)
                else pd.NaT
            ),
            "selected_last_date": (
                pd.Timestamp(selected_dates.max()).normalize()
                if len(selected_dates)
                else pd.NaT
            ),
            **metrics,
        })

        if is_baseline:
            baseline_rows = tenor_frame.loc[mask].copy()
            baseline_rows["layer"] = layer
            baseline_rows["sleeve_id"] = f"{layer}_{tenor}D"
            baseline_rows["vrp_log_threshold"] = params["vrp_log"]
            baseline_rows["z_threshold"] = params["z"]
            baseline_rows["rsi_cap"] = params["rsi_cap"]
            baseline_rows["rv21d_floor"] = params["rv_floor"]
            baseline_opportunity_parts.append(baseline_rows)

    if baseline_hits != 1:
        raise RuntimeError(
            f"{layer} {tenor}D baseline appeared {baseline_hits} times in its parameter grid."
        )

    print(
        f"[{sleeve_number:02d}/{len(BASELINES):02d}] "
        f"{layer:<9} {tenor:>2}D complete — "
        f"{len(grid):,} parameter sets, {eligible_rows:,} eligible outcome rows"
    )

results = pd.DataFrame(result_rows)

if len(results) != EXPECTED_RESULT_ROWS:
    raise RuntimeError(
        f"Unexpected result row count: {len(results):,} != {EXPECTED_RESULT_ROWS:,}"
    )

if int(results["is_baseline"].sum()) != len(BASELINES):
    raise RuntimeError(
        f"Expected {len(BASELINES)} baseline rows; found {int(results['is_baseline'].sum())}"
    )

results = assign_parameter_ranks(results)

baseline_opportunities = pd.concat(
    baseline_opportunity_parts,
    ignore_index=True,
) if baseline_opportunity_parts else pd.DataFrame()

# This is a diagnostic proving that simultaneous opportunities remain present.
baseline_sleeves_per_date = (
    baseline_opportunities.groupby("trade_date")["sleeve_id"].nunique()
    if len(baseline_opportunities)
    else pd.Series(dtype=int)
)
max_baseline_sleeves_per_date = (
    int(baseline_sleeves_per_date.max())
    if len(baseline_sleeves_per_date)
    else 0
)

print("=" * 100)
print("Cell 3 — Parameter sweep complete")
print("=" * 100)
print(f"Result rows:                         {len(results):,}")
print(f"Baseline rows:                       {int(results['is_baseline'].sum()):,}")
print(f"Ranking-eligible rows:               {int(results['ranking_eligible'].sum()):,}")
print(f"Baseline opportunity rows:           {len(baseline_opportunities):,}")
print(f"Max baseline sleeves on one date:    {max_baseline_sleeves_per_date}")
print()
print("No one-trade-per-day filter was applied.")
print("\nCELL 3 COMPLETE")

[01/13] Core      21D complete — 1,925 parameter sets, 1,358 eligible outcome rows
[02/13] Core      24D complete — 1,925 parameter sets, 1,357 eligible outcome rows
[03/13] Core      27D complete — 1,925 parameter sets, 1,354 eligible outcome rows
[04/13] Core      30D complete — 1,925 parameter sets, 1,354 eligible outcome rows
[05/13] Core      33D complete — 1,925 parameter sets, 1,351 eligible outcome rows
[06/13] Secondary 12D complete — 1,925 parameter sets, 1,365 eligible outcome rows
[07/13] Secondary 15D complete — 1,925 parameter sets, 1,363 eligible outcome rows
[08/13] Secondary 18D complete — 1,925 parameter sets, 1,361 eligible outcome rows
[09/13] Secondary 21D complete — 1,925 parameter sets, 1,358 eligible outcome rows
[10/13] Secondary 24D complete — 1,925 parameter sets, 1,357 eligible outcome rows
[11/13] Secondary 27D complete — 1,925 parameter sets, 1,354 eligible outcome rows
[12/13] Secondary 30D complete — 1,925 parameter sets, 1,354 eligible outcome rows
[13/

In [5]:
# ======================================================================================
# Cell 4 — Build leaderboard, save outputs, and display review tables
# ======================================================================================

leaderboard_parts = []

for (layer, tenor), group in results.groupby(["layer", "tenor"], sort=False):
    top = group.loc[group["ranking_eligible"]].nsmallest(
        TOP_PER_SLEEVE,
        "parameter_set_rank",
    )
    baseline = group.loc[group["is_baseline"]]
    combined = pd.concat([top, baseline], ignore_index=True)
    combined = combined.drop_duplicates("parameter_set_id", keep="first")
    leaderboard_parts.append(combined)

leaderboard = (
    pd.concat(leaderboard_parts, ignore_index=True)
    .sort_values(
        ["layer_sort", "tenor", "is_baseline", "parameter_set_rank"],
        ascending=[True, True, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

baseline_results = (
    results.loc[results["is_baseline"]]
    .sort_values(["layer_sort", "tenor"])
    .reset_index(drop=True)
)

top_candidates = (
    results.loc[results["parameter_set_rank"].eq(1)]
    .sort_values(["layer_sort", "tenor"])
    .reset_index(drop=True)
)

if len(baseline_results) != len(BASELINES):
    raise RuntimeError(
        f"Expected {len(BASELINES)} baseline result rows, found {len(baseline_results)}"
    )

# Some sleeves may have no rank-eligible set if every candidate has fewer than 30 trades.
expected_top_count = int(
    results.groupby(["layer", "tenor"])["ranking_eligible"].any().sum()
)
if len(top_candidates) != expected_top_count:
    raise RuntimeError(
        f"Expected {expected_top_count} top-ranked rows, found {len(top_candidates)}"
    )

comparison_cols = [
    "layer",
    "tenor",
    "parameter_set_id",
    "vrp_log_threshold",
    "z_3m_threshold",
    "z_1y_threshold",
    "rsi_cap",
    "rv21d_floor",
    "trade_count",
    "frequency",
    "win_rate",
    "avg_return",
    "avg_win",
    "avg_loss",
    "max_loss",
    "tail_loss",
    "worst_1pct_loss",
    "profit_factor",
    "max_drawdown",
    "largest_losing_cluster",
    "largest_losing_cluster_count",
    "avg_pnl_per_day",
    "worst_20_trade_pnl_dollars",
    "parameter_quality_score",
    "parameter_set_rank",
]

baseline_review = baseline_results[comparison_cols].copy()
top_review = top_candidates[comparison_cols].copy()

baseline_review["result_type"] = "BASELINE"
top_review["result_type"] = "TOP_RANKED"

baseline_vs_top = pd.concat(
    [baseline_review, top_review],
    ignore_index=True,
).sort_values(
    ["layer", "tenor", "result_type"],
    kind="mergesort",
)

baseline_csv = (
    OUT_AUDIT_DIR
    / f"parameter_sweep_baseline_results_{RUN_TIMESTAMP}.csv"
)
top_csv = (
    OUT_AUDIT_DIR
    / f"parameter_sweep_top_candidates_{RUN_TIMESTAMP}.csv"
)
join_csv = (
    OUT_AUDIT_DIR
    / f"parameter_sweep_join_coverage_{RUN_TIMESTAMP}.csv"
)
opportunity_csv = (
    OUT_AUDIT_DIR
    / f"parameter_sweep_baseline_opportunities_{RUN_TIMESTAMP}.csv"
)
manifest_path = (
    OUT_AUDIT_DIR
    / f"parameter_sweep_manifest_{RUN_TIMESTAMP}.json"
)

# Explicit production-path protection.
production_dir = (
    PROJECT_ROOT / "data" / "processed" / "vrp_final_signal"
).resolve()

for output_path in [RESULTS_PATH, LEADERBOARD_PATH]:
    resolved = output_path.resolve()
    try:
        resolved.relative_to(production_dir)
    except ValueError:
        pass
    else:
        raise RuntimeError(
            f"Refusing to write parameter-sweep output inside production directory: {resolved}"
        )

results.to_parquet(RESULTS_PATH, index=False)
leaderboard.to_parquet(LEADERBOARD_PATH, index=False)

baseline_results.to_csv(baseline_csv, index=False)
top_candidates.to_csv(top_csv, index=False)
join_coverage.to_csv(join_csv, index=False)
baseline_opportunities.to_csv(opportunity_csv, index=False)

manifest = {
    "notebook_version": "vrp_wilder_rsi_tenor_isolated_parameter_sweep_v1",
    "created_timestamp": RUN_TIMESTAMP,
    "project_root": str(PROJECT_ROOT),
    "signal_source": str(SIGNAL_PATH),
    "outcome_source": str(OUTCOME_PATH),
    "outcome_model_spec": LOCKED_MODEL_SPEC,
    "outcome_fields": [
        "normalized_pnl_pct",
        "normalized_pnl_dollars",
        "win",
        "last_forward_rv_date",
    ],
    "legacy_outcome_rsi_used": False,
    "rsi_formula_version": ACCEPTED_RSI_VERSION,
    "target_tenors": TARGET_TENORS,
    "inactive_tenors": [9],
    "inactive_layers": ["Core Front"],
    "active_sleeves": [
        {"layer": layer, "tenor": tenor, **baseline}
        for (layer, tenor), baseline in BASELINES.items()
    ],
    "parameter_grid": {
        "vrp_offsets": VRP_OFFSETS,
        "z_offsets": Z_OFFSETS,
        "rsi_offsets": RSI_OFFSETS,
        "rv21d_offsets": RV_OFFSETS,
        "sets_per_sleeve": EXPECTED_GRID_SIZE_PER_SLEEVE,
    },
    "selection_rule": "none_all_qualifying_tenors_preserved",
    "one_trade_per_day_applied": False,
    "minimum_ranking_trades": MIN_RANKING_TRADES,
    "tail_fraction": TAIL_FRACTION,
    "worst_fraction": WORST_FRACTION,
    "result_rows": int(len(results)),
    "leaderboard_rows": int(len(leaderboard)),
    "baseline_rows": int(len(baseline_results)),
    "top_ranked_rows": int(len(top_candidates)),
    "joined_usable_rows": int(len(joined_usable)),
    "max_baseline_sleeves_per_date": max_baseline_sleeves_per_date,
    "outputs": {
        "results_parquet": str(RESULTS_PATH),
        "leaderboard_parquet": str(LEADERBOARD_PATH),
        "baseline_csv": str(baseline_csv),
        "top_candidates_csv": str(top_csv),
        "join_coverage_csv": str(join_csv),
        "baseline_opportunities_csv": str(opportunity_csv),
    },
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

# Reopen the two processed artifacts and validate the written files.
results_check = pd.read_parquet(RESULTS_PATH)
leaderboard_check = pd.read_parquet(LEADERBOARD_PATH)

if len(results_check) != EXPECTED_RESULT_ROWS:
    raise RuntimeError(
        f"Written results row count mismatch: {len(results_check):,}"
    )
if int(results_check["is_baseline"].sum()) != len(BASELINES):
    raise RuntimeError("Written results do not contain exactly one baseline per sleeve.")
if results_check.duplicated(
    ["layer", "tenor", "parameter_set_id"]
).any():
    raise RuntimeError("Written results contain duplicate sleeve/parameter-set rows.")
if leaderboard_check.empty:
    raise RuntimeError("Written leaderboard is empty.")

print("=" * 100)
print("Cell 4 — Outputs written and validated")
print("=" * 100)
print(f"Results:                {RESULTS_PATH}")
print(f"Leaderboard:            {LEADERBOARD_PATH}")
print(f"Baseline audit:         {baseline_csv}")
print(f"Top-candidate audit:    {top_csv}")
print(f"Join-coverage audit:    {join_csv}")
print(f"Baseline opportunities: {opportunity_csv}")
print(f"Manifest:               {manifest_path}")
print()
print("Canonical production final-signal files were not modified.")

print("\nBaseline versus top-ranked candidate:")
display(baseline_vs_top[[
    "result_type",
    "layer",
    "tenor",
    "vrp_log_threshold",
    "z_3m_threshold",
    "z_1y_threshold",
    "rsi_cap",
    "rv21d_floor",
    "trade_count",
    "frequency",
    "win_rate",
    "avg_return",
    "avg_win",
    "avg_loss",
    "tail_loss",
    "worst_1pct_loss",
    "profit_factor",
    "max_drawdown",
    "largest_losing_cluster",
    "avg_pnl_per_day",
    "parameter_quality_score",
    "parameter_set_rank",
]])

print("\nPASS — tenor-isolated repaired-Wilder-RSI parameter sweep completed.")
print("No one-trade-per-day filter, sizing step, or production overwrite was performed.")

Cell 4 — Outputs written and validated
Results:                C:\Users\patri\vrp_project\data\processed\vrp_wilder_rsi_parameter_sweep\vrp_wilder_rsi_tenor_isolated_parameter_sweep_results_v1.parquet
Leaderboard:            C:\Users\patri\vrp_project\data\processed\vrp_wilder_rsi_parameter_sweep\vrp_wilder_rsi_tenor_isolated_parameter_leaderboard_v1.parquet
Baseline audit:         C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep\parameter_sweep_baseline_results_20260710_151253.csv
Top-candidate audit:    C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep\parameter_sweep_top_candidates_20260710_151253.csv
Join-coverage audit:    C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep\parameter_sweep_join_coverage_20260710_151253.csv
Baseline opportunities: C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep\parameter_sweep_baseline_opportunities_20260710_151253.csv
Manifest: 

,result_type,layer,tenor,vrp_log_threshold,z_3m_threshold,z_1y_threshold,rsi_cap,rv21d_floor,trade_count,frequency,win_rate,avg_return,avg_win,avg_loss,tail_loss,worst_1pct_loss,profit_factor,max_drawdown,largest_losing_cluster,avg_pnl_per_day,parameter_quality_score,parameter_set_rank
0,BASELINE,Core,21,0.65,0.7,0.7,70.0,8.5,145,0.106775,0.848276,0.012258,0.019613,-0.028865,-0.054265,-0.082642,3.798860,-0.355922,-0.355922,583.700913,0.419654,1111
13,TOP_RANKED,Core,21,0.65,0.9,0.9,68.0,7.0,119,0.087629,0.882353,0.013814,0.020138,-0.033618,-0.056900,-0.082642,4.492689,-0.323953,-0.323953,657.807097,0.880649,1
1,BASELINE,Core,24,0.65,0.7,0.7,70.0,8.5,147,0.108327,0.850340,0.012623,0.020123,-0.029992,-0.054491,-0.074949,3.812146,-0.356134,-0.324563,525.945779,0.561948,771
14,TOP_RANKED,Core,24,0.65,0.9,0.9,70.0,10.0,111,0.081798,0.873874,0.014616,0.021755,-0.034847,-0.057201,-0.074949,4.325447,-0.308135,-0.276564,608.991415,0.892987,1
2,BASELINE,Core,27,0.70,0.7,0.7,70.0,8.5,135,0.099705,0.874074,0.015415,0.022927,-0.036728,-0.060046,-0.092584,4.332992,-0.287049,-0.274703,570.933231,0.350563,1267
15,TOP_RANKED,Core,27,0.80,0.8,0.8,70.0,7.0,117,0.086411,0.897436,0.017652,0.023152,-0.030472,-0.045962,-0.074949,6.648146,-0.165617,-0.153271,653.794911,0.952035,1
3,BASELINE,Core,30,0.70,0.7,0.7,70.0,8.5,133,0.098227,0.902256,0.018041,0.024222,-0.039011,-0.058587,-0.098130,5.731477,-0.226445,-0.226445,601.380666,0.521732,830
16,TOP_RANKED,Core,30,0.80,0.7,0.7,74.0,7.5,115,0.084934,0.930435,0.020730,0.024735,-0.032844,-0.038883,-0.072909,10.072923,-0.120657,-0.120657,690.993859,0.936926,1
4,BASELINE,Core,33,0.70,0.7,0.7,70.0,8.5,135,0.099926,0.925926,0.020213,0.024832,-0.037519,-0.049042,-0.081805,8.273006,-0.173716,-0.173716,612.523672,0.541905,878
17,TOP_RANKED,Core,33,0.80,0.9,0.9,69.0,7.0,107,0.079201,0.962617,0.023031,0.025884,-0.050431,-0.031296,-0.081805,13.216153,-0.103671,-0.103671,697.904711,0.939177,1



PASS — tenor-isolated repaired-Wilder-RSI parameter sweep completed.
No one-trade-per-day filter, sizing step, or production overwrite was performed.
